# RAG Evaluation Pipeline - DORIS
## RAGAS + GraphRAG-Bench Metrics (arXiv 2506.05690)

Stack:
- Dataset: RAGAS testset ODS (Polish)
- LLM: `gpt-4o-mini` (OpenAI key from env)
- Embeddings: `text-embedding-3-small

Metrics:
| Category | Metrics |
|-----------|---------|
| Graph Quality (§3.3 + Table 11) | Node/Edge Count, Avg Degree, Avg Clustering, Prop. Non-isolated, Component Size |
| Retrieval (§3.3) | Context Relevance Eq.(3), Evidence Recall Eq.(4) |
| Generation (App. D) | ROUGE-L, Answer Accuracy Eq.(5-6), Faithfulness Eq.(7), Evidence Coverage Eq.(8) |
| RAGAS | Faithfulness, ResponseRelevancy, ContextPrecision, ContextRecall, FactualCorrectness, SemanticSimilarity |


In [ ]:
# %pip install ragas networkx rouge-score sentence-transformers openai neo4j pandas odfpy matplotlib seaborn

In [ ]:
import os, warnings
from pathlib import Path
from typing import List, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dotenv import load_dotenv
from rouge_score import rouge_scorer
from sentence_transformers import util
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness, ResponseRelevancy,
    LLMContextPrecisionWithReference, LLMContextRecall,
    FactualCorrectness, SemanticSimilarity, RougeScore,
)
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings("ignore")
load_dotenv()

## Configuration

In [ ]:
import tiktoken

TESTSET_PATH = Path("Graph RAG Dataset/testset/doris_qa_results_enriched.csv")

os.environ.get("OPENAI_API_KEY")

LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
MAX_EMB_TOKENS = 8000
_enc = tiktoken.encoding_for_model(EMBEDDING_MODEL)

## Loading models

In [ ]:
from openai import OpenAI
from ragas.embeddings import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings as LCOpenAIEmbeddings

# LLM (RAGAS + LLM-based scoring)
evaluator_llm = llm_factory(LLM_MODEL, client=OpenAI())
print(f"LLM: {LLM_MODEL}")

lc_emb = OpenAIEmbeddings(model=EMBEDDING_MODEL, client=OpenAI())
evaluator_emb = LangchainEmbeddingsWrapper(LCOpenAIEmbeddings(model=EMBEDDING_MODEL))
print("Embeddings wrapper ready")

In [ ]:
def truncate_for_embedding(text: str, max_tokens: int = MAX_EMB_TOKENS) -> str:
    tokens = _enc.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return _enc.decode(tokens[:max_tokens])

def embed_texts_safe(texts: List[str]) -> list:
    truncated = [truncate_for_embedding(t) for t in texts if t and t.strip()]
    if not truncated:
        return []
    return lc_emb.embed_texts(truncated)

## Loading the RAGAS dataset

Columns from `TestsetGenerator`: `user_input`, `reference_contexts`, `reference`


In [ ]:
import json, ast, re


df_raw = pd.read_csv(TESTSET_PATH)
print(f"Loaded {len(df_raw)} rows | columns: {list(df_raw.columns)}")


def parse_source_texts(raw) -> list[str]:
    """Parses the source_texts column: a list of JSONs with a 'text' key."""
    if isinstance(raw, list):
        return [item["text"] for item in raw if isinstance(item, dict) and "text" in item]
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return [item["text"] for item in parsed if isinstance(item, dict) and "text" in item]
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [item["text"] for item in parsed if isinstance(item, dict) and "text" in item]
    except Exception:
        pass
    return [raw]  # fallback: the whole string as one context


def parse_contexts(raw) -> list[str]:
    """Parses reference_contexts (a list of strings or JSON)."""
    if isinstance(raw, list):
        return [str(x) for x in raw if x]
    if isinstance(raw, str):
        raw = raw.strip()
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(raw)
                if isinstance(parsed, list):
                    return [str(x) for x in parsed if x]
            except Exception:
                pass
        return [raw] if raw else []
    return []


df_ok = df_raw[df_raw["status"].fillna("").str.lower() != "error"].copy()
print(f"Rows without errors: {len(df_ok)}/{len(df_raw)}")

rows_doris = []
for _, row in df_ok.iterrows():
    question   = str(row["user_input"]).strip()
    reference  = str(row["reference"]).strip()
    response   = str(row["answer"]).strip() if pd.notna(row.get("answer")) else ""
    ref_ctx    = parse_contexts(row.get("reference_contexts", ""))
    ret_ctx    = parse_source_texts(row.get("source_texts", ""))

    if question and reference:
        rows_doris.append({
            "question":           question,
            "reference":          reference,
            "response":           response,
            "reference_contexts": ref_ctx,
            "retrieved_contexts": ret_ctx,
        })

qa_results = {"doris": rows_doris}

print(f"\nLoaded {len(rows_doris)} samples for mode 'doris'")
r0 = rows_doris[0]
print(f"  question          : {r0['question'][:80]}...")
print(f"  reference         : {r0['reference'][:80]}...")
print(f"  response          : {r0['response'][:80]}...")
print(f"  reference_contexts: {len(r0['reference_contexts'])} fragments")
print(f"  retrieved_contexts: {len(r0['retrieved_contexts'])} fragments")

In [ ]:
# Check the distribution of fragment counts
import numpy as np

ref_lens = [len(r["reference_contexts"]) for r in rows_doris]
ret_lens = [len(r["retrieved_contexts"]) for r in rows_doris]

print("reference_contexts:")
print(f"  min={min(ref_lens)}, max={max(ref_lens)}, avg={np.mean(ref_lens):.1f}")
print(f"  empty (0): {sum(1 for x in ref_lens if x == 0)}")

print("\nretrieved_contexts (source_texts):")
print(f"  min={min(ret_lens)}, max={max(ret_lens)}, avg={np.mean(ret_lens):.1f}")
print(f"  empty (0): {sum(1 for x in ret_lens if x == 0)}")

# Example of a raw reference_contexts value
print("\nExample of raw reference_contexts (row 0):")
print(repr(df_ok["reference_contexts"].iloc[0][:300]))

In [ ]:
MODES  = list(qa_results.keys())          # ["doris"]
COLORS = {"doris": "#7B2D8B"}

## Retrieval Metrics (Paper Eq. 3 & 4)

Uses the embedding model.


In [ ]:
RETRIEVAL_THRESHOLD = 0.65   # threshold for Evidence Recall
RELEVANCE_THRESHOLD_HIGH = 0.72
RELEVANCE_THRESHOLD_MID = 0.45

def score_to_rating(score: float) -> float:
    if score >= RELEVANCE_THRESHOLD_HIGH: return 1.0
    elif score >= RELEVANCE_THRESHOLD_MID: return 0.5
    return 0.0

def compute_context_relevance(question: str, retrieved_contexts: List[str]) -> Dict:
    """Context Relevance — Eq.(3) from arXiv 2506.05690."""
    if isinstance(retrieved_contexts, str):
        retrieved_contexts = [retrieved_contexts] if retrieved_contexts.strip() else []
    if not retrieved_contexts:
        return {"context_relevance": 0.0, "r1": 0.0, "r2": 0.0}

    valid_contexts = [c for c in retrieved_contexts if c and c.strip()]
    if not valid_contexts:
        return {"context_relevance": 0.0, "r1": 0.0, "r2": 0.0}

    q_emb = embed_texts_safe([question])
    c_emb = embed_texts_safe(valid_contexts)
    scores = util.cos_sim(q_emb, c_emb)[0].tolist()
    top2 = sorted(scores, reverse=True)[:2]
    while len(top2) < 2: top2.append(0.0)
    r1, r2 = score_to_rating(top2[0]), score_to_rating(top2[1])
    return {
        "context_relevance": round((r1 + r2) / 2, 4),
        "r1": r1, "r2": r2,
        "top_scores": [round(s, 4) for s in top2],
    }

def compute_evidence_recall(reference_contexts: List[str], retrieved_contexts: List[str]) -> Dict:
    """Evidence Recall — Eq.(4) from arXiv 2506.05690."""
    if isinstance(retrieved_contexts, str):
        retrieved_contexts = [retrieved_contexts] if retrieved_contexts.strip() else []

    reference_contexts = [c for c in reference_contexts if c and c.strip()]
    retrieved_contexts = [c for c in retrieved_contexts if c and c.strip()]

    if not reference_contexts:
        return {"evidence_recall": 0.0, "matched": 0, "total": 0}
    if not retrieved_contexts:
        return {"evidence_recall": 0.0, "matched": 0, "total": len(reference_contexts)}
    ref_emb = embed_texts_safe(reference_contexts)
    ret_emb = embed_texts_safe(retrieved_contexts)
    sim = util.cos_sim(ref_emb, ret_emb)
    matched = sum(1 for i in range(len(reference_contexts))
                  if sim[i].max().item() >= RETRIEVAL_THRESHOLD)
    return {
        "evidence_recall": round(matched / len(reference_contexts), 4),
        "matched": matched,
        "total": len(reference_contexts),
    }

retrieval_dfs = {}
for mode, rows in qa_results.items():
    results = []
    for r in rows:
        cr = compute_context_relevance(r["question"], r["retrieved_contexts"])
        er = compute_evidence_recall(r["reference_contexts"], r["retrieved_contexts"])
        results.append({
            "question": r["question"][:55] + "...",
            "mode":     mode,
            **cr, **er,
        })
    retrieval_dfs[mode] = pd.DataFrame(results)

print("=" * 65)
print("  RETRIEVAL METRICS - Avg per mode")
print("=" * 65)
print(f"  {'Mode':<12} {'Context Relevance':>20} {'Evidence Recall':>18}")
print("  " + "-"*52)
for mode, df in retrieval_dfs.items():
    print(f"  {mode:<12} {df['context_relevance'].mean():>20.4f} {df['evidence_recall'].mean():>18.4f}")

## Generation Accuracy Metrics (App. D — Eq. 5–8)


In [ ]:
CLAIM_THRESHOLD   = 0.62
COVERAGE_THRESHOLD = 0.58

_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)


def atomic_claims(text: str) -> List[str]:
    parts = re.split(r"[.;!?]", text)
    return [p.strip() for p in parts if len(p.strip()) > 8]


def rouge_l(pred: str, ref: str) -> float:
    return round(_rouge.score(ref, pred)["rougeL"].fmeasure, 4)


def semantic_similarity(response: str, reference: str) -> float:
    """SS = cos(f_i, c_j) — Eq.(6)."""
    embs = embed_texts_safe([response, reference])
    return round(util.cos_sim(embs[0], embs[1]).item(), 4)


def factual_correctness(response: str, reference: str) -> Dict:
    """FC = 2*TP/(TP+FP+FN) — Eq.(6)."""
    rc = atomic_claims(response)
    rf = atomic_claims(reference)
    if not rc or not rf:
        return {"factual_correctness": 0.0, "TP": 0, "FP": len(rc), "FN": len(rf)}
    re_embs = embed_texts_safe(rc)
    rf_embs = embed_texts_safe(rf)
    sim = util.cos_sim(re_embs, rf_embs)
    TP = sum(1 for i in range(len(rc)) if sim[i].max().item() >= CLAIM_THRESHOLD)
    FP = len(rc) - TP
    FN = sum(1 for j in range(len(rf)) if sim[:, j].max().item() < CLAIM_THRESHOLD)
    fc = 2*TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0
    return {"factual_correctness": round(fc, 4), "TP": TP, "FP": FP, "FN": FN}


def answer_accuracy(response: str, reference: str, alpha: float = 0.5) -> Dict:
    """AC = alpha*FC + (1-alpha)*SS — Eq.(5)."""
    fc = factual_correctness(response, reference)
    ss = semantic_similarity(response, reference)
    ac = alpha * fc["factual_correctness"] + (1 - alpha) * ss
    return {"answer_accuracy": round(ac, 4), "factual_correctness": fc["factual_correctness"],
            "semantic_similarity": ss}


def faithfulness_score(response: str, retrieved_contexts: List[str]) -> Dict:
    """FS = |{c∈A: S(c,C)}|/|A| — Eq.(7)."""
    if isinstance(retrieved_contexts, str):
        retrieved_contexts = [retrieved_contexts] if retrieved_contexts.strip() else []

    claims = atomic_claims(response)
    if not claims or not retrieved_contexts:
        return {"faithfulness_paper": 0.0, "supported": 0, "total_claims": len(claims)}
    ctx_emb = embed_texts_safe([" ".join(retrieved_contexts)])
    cl_embs = embed_texts_safe(claims)
    sims = util.cos_sim(cl_embs, ctx_emb)[:, 0].tolist()
    supported = sum(1 for s in sims if s >= CLAIM_THRESHOLD)
    return {"faithfulness_paper": round(supported/len(claims), 4),
            "supported": supported, "total_claims": len(claims)}


def evidence_coverage(response: str, reference_contexts: List[str]) -> Dict:
    """Cov = |{e∈E: M(e,G)}|/|E| — Eq.(8)."""
    if not reference_contexts:
        return {"evidence_coverage": 0.0}
    resp_emb = embed_texts_safe([response])
    ref_embs = embed_texts_safe(reference_contexts)
    sims = util.cos_sim(ref_embs, resp_emb)[:, 0].tolist()
    covered = sum(1 for s in sims if s >= COVERAGE_THRESHOLD)
    return {"evidence_coverage": round(covered/len(reference_contexts), 4),
            "covered": covered, "total": len(reference_contexts)}


# For each mode
generation_dfs = {}
for mode, rows in qa_results.items():
    gen_rows = []
    for r in rows:
        if not r["response"]:
            gen_rows.append({"question": r["question"][:55]+"...", "mode": mode,
                             "rouge_l": 0, "answer_accuracy": 0, "factual_correctness": 0,
                             "semantic_similarity": 0, "faithfulness_paper": 0, "evidence_coverage": 0})
            continue
        row = {"question": r["question"][:55]+"...", "mode": mode}
        row["rouge_l"] = rouge_l(r["response"], r["reference"])
        row.update(answer_accuracy(r["response"], r["reference"]))
        row.update(faithfulness_score(r["response"], r["retrieved_contexts"]))
        row.update(evidence_coverage(r["response"], r["reference_contexts"]))
        gen_rows.append(row)
    generation_dfs[mode] = pd.DataFrame(gen_rows)

# Summary table
summary_gen = pd.DataFrame([
    {
        "mode": mode,
        "rouge_l":            df["rouge_l"].mean(),
        "answer_accuracy":    df["answer_accuracy"].mean(),
        "factual_correctness":df["factual_correctness"].mean(),
        "semantic_similarity":df["semantic_similarity"].mean(),
        "faithfulness_paper": df["faithfulness_paper"].mean(),
        "evidence_coverage":  df["evidence_coverage"].mean(),
    }
    for mode, df in generation_dfs.items()
]).round(4)

print("=" * 85)
print("GENERATION METRICS - Avg per mode")
print("=" * 85)
print(summary_gen.to_string(index=False))

## RAGAS Metrics

In [ ]:
def to_ctx_list(ctx) -> List[str]:
    if isinstance(ctx, list):
        return ctx if ctx else ["brak kontekstu"]
    if isinstance(ctx, str) and ctx.strip():
        return [ctx]
    return ["brak kontekstu"]

ragas_metric_list = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_emb),
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm),
    SemanticSimilarity(embeddings=evaluator_emb),
    RougeScore(),
]

ragas_dfs = {}
for mode, rows in qa_results.items():
    print(f"RAGAS [{mode}]...")
    samples_eval = [
        SingleTurnSample(
            user_input=r["question"],
            response=r["response"] or "brak odpowiedzi",
            reference=r["reference"],
            retrieved_contexts=to_ctx_list(r["retrieved_contexts"]),
            reference_contexts=to_ctx_list(r["reference_contexts"]),
        )
        for r in rows
    ]
    ds = EvaluationDataset(samples=samples_eval)
    result = evaluate(dataset=ds, metrics=ragas_metric_list)
    df_r = result.to_pandas()
    df_r["mode"] = mode
    ragas_dfs[mode] = df_r
    print(f"  {mode} OK")


first_df = next(iter(ragas_dfs.values()))
ragas_num_cols = [c for c in first_df.columns
                  if c not in ["user_input","response","retrieved_contexts",
                               "reference","reference_contexts","mode"]
                  and pd.api.types.is_numeric_dtype(first_df[c])]

summary_ragas = pd.DataFrame([
    {"mode": mode, **{col: df[col].mean() for col in ragas_num_cols}}
    for mode, df in ragas_dfs.items()
]).round(4)

print("\n" + "=" * 75)
print("RAGAS METRICS - Avg per mode")
print("=" * 75)
print(summary_ragas.to_string(index=False))

In [ ]:
import csv

all_ragas = pd.concat(ragas_dfs.values(), ignore_index=True)
all_ragas.to_csv(
    "Graph RAG Dataset/testset/doris_ragas_detailed.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

summary_ragas.to_csv(
    "Graph RAG Dataset/testset/doris_ragas_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved {len(all_ragas)} detailed rows + summary")

## Full Comparison Report (GRAPH vs VANILLA vs HYBRID)

In [ ]:
all_results = []
for mode in qa_results.keys():
    df_r = retrieval_dfs[mode].copy()
    df_g = generation_dfs[mode].copy()
    df_ra = ragas_dfs[mode].copy()

    base = df_r[["question","mode","context_relevance","evidence_recall"]].reset_index(drop=True)
    base["answer"] = [r["response"] for r in qa_results[mode]]

    gen = df_g[["rouge_l","answer_accuracy","factual_correctness",
                "semantic_similarity","faithfulness_paper","evidence_coverage"]].reset_index(drop=True)
    rag = df_ra[ragas_num_cols].add_prefix("ragas_").reset_index(drop=True)
    combined = pd.concat([base, gen, rag], axis=1)
    all_results.append(combined)

df_all = pd.concat(all_results, ignore_index=True)

metric_cols = [c for c in df_all.columns if c not in ["question", "answer", "mode"]]
pivot = df_all.groupby("mode")[metric_cols].mean().round(4).T
pivot.index.name = "Metric"

def source_of(m):
    if m in ["context_relevance","evidence_recall"]:
        return "Paper Retrieval"
    if m in ["rouge_l","answer_accuracy","factual_correctness",
              "semantic_similarity","faithfulness_paper","evidence_coverage"]:
        return "Paper Generation"
    return "RAGAS"

pivot.insert(0, "Source", [source_of(m) for m in pivot.index])
pivot = pivot.sort_values("Source")

print("=" * 90)
print("FULL COMPARISON: Paper + RAGAS  |  DORIS")
print("=" * 90)
print(pivot.to_string())
df_all

## Visualizations

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Context Relevance per sample
ax2 = fig.add_subplot(gs[0, 0])
vals = retrieval_dfs["doris"]["context_relevance"].values
ax2.plot(vals, "o-", color=COLORS["doris"], markersize=4, alpha=0.8, label="doris")
ax2.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax2.set_title("Context Relevance per sample", fontweight="bold", fontsize=9)
ax2.set_xlabel("Sample idx"); ax2.set_ylabel("Score"); ax2.set_ylim(-0.05, 1.15)

# Evidence Recall per sample
ax3 = fig.add_subplot(gs[0, 1])
vals = retrieval_dfs["doris"]["evidence_recall"].values
ax3.plot(vals, "s-", color=COLORS["doris"], markersize=4, alpha=0.8, label="doris")
ax3.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax3.set_title("Evidence Recall per sample", fontweight="bold", fontsize=9)
ax3.set_xlabel("Sample idx"); ax3.set_ylabel("Score"); ax3.set_ylim(-0.05, 1.15)

# Generation bar
ax4 = fig.add_subplot(gs[0, 2])
gen_m = ["rouge_l", "answer_accuracy", "faithfulness_paper", "evidence_coverage"]
vals  = [generation_dfs["doris"][m].mean() for m in gen_m]
ax4.bar(gen_m, vals, color=COLORS["doris"], alpha=0.85)
ax4.set_xticklabels(["ROUGE-L","Ans.Acc","Faithful","EvCov"], fontsize=8)
ax4.set_title("Generation Metrics (avg) – DORIS", fontweight="bold", fontsize=9)
ax4.set_ylim(0, 1.2); ax4.set_ylabel("Score")

# RAGAS heatmap
ax5 = fig.add_subplot(gs[1, :2])
hm_data = summary_ragas.set_index("mode")[ragas_num_cols]
sns.heatmap(hm_data.T, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, ax=ax5, linewidths=0.5, annot_kws={"size": 9})
ax5.set_title("RAGAS Metrics Heatmap – DORIS", fontweight="bold", fontsize=10)
ax5.set_xlabel("RAG Mode")

# Paper vs RAGAS
ax6 = fig.add_subplot(gs[1, 2])
paper_ragas = {
    "faithfulness":   ("faithfulness_paper", "ragas_faithfulness"),
    "ev. recall":     ("evidence_recall",     "ragas_context_recall"),
    "rouge-l":        ("rouge_l",             "ragas_rouge_score"),
    "sem. sim.":      ("semantic_similarity",  "ragas_semantic_similarity"),
}
labels_, paper_vals, ragas_vals = [], [], []
for label, (pk, rk) in paper_ragas.items():
    if pk in df_all.columns and rk in df_all.columns:
        labels_.append(label)
        paper_vals.append(df_all[pk].mean())
        ragas_vals.append(df_all[rk].mean())

x_ = np.arange(len(labels_))
ax6.bar(x_ - 0.2, paper_vals, 0.36, label="Paper", color="#4C72B0", alpha=0.85)
ax6.bar(x_ + 0.2, ragas_vals, 0.36, label="RAGAS",  color="#DD8452", alpha=0.85)
for i, (pv, rv) in enumerate(zip(paper_vals, ragas_vals)):
    ax6.text(i-0.2, pv+0.02, f"{pv:.2f}", ha="center", fontsize=8, color="#4C72B0", fontweight="bold")
    ax6.text(i+0.2, rv+0.02, f"{rv:.2f}", ha="center", fontsize=8, color="#DD8452", fontweight="bold")
ax6.set_xticks(x_); ax6.set_xticklabels(labels_, fontsize=8)
ax6.set_ylim(0, 1.2); ax6.set_title("Paper vs RAGAS (DORIS avg)", fontweight="bold", fontsize=9)
ax6.legend(fontsize=8)

fig.suptitle("DORIS - RAG Evaluation Dashboard", fontsize=13, fontweight="bold")
plt.savefig("doris_eval_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## Exporting results

In [ ]:
output_dir = TESTSET_PATH.parent
xlsx_path  = output_dir / "doris_evaluation_results.xlsx"

with pd.ExcelWriter(xlsx_path) as writer:
    # Per-sample results for each mode
    for mode in MODES:
        df_r  = retrieval_dfs[mode]
        df_g  = generation_dfs[mode]
        df_ra = ragas_dfs[mode][["user_input"] + ragas_num_cols].rename(
            columns={c: "ragas_"+c for c in ragas_num_cols})
        full = pd.concat([
            df_r.reset_index(drop=True),
            df_g.drop(columns=["question","mode"], errors="ignore").reset_index(drop=True),
            df_ra.reset_index(drop=True),
        ], axis=1)
        full.to_excel(writer, sheet_name=f"mode_{mode}", index=False)

    pivot.to_excel(writer, sheet_name="Summary_pivot")
    summary_ragas.to_excel(writer, sheet_name="RAGAS_summary", index=False)
    summary_gen.to_excel(writer, sheet_name="Generation_summary", index=False)

print(f"Results saved: {xlsx_path}")


print()
print("=" * 65)
print("FINAL SUMMARY - DORIS RAG Evaluation")
print("=" * 65)
print(f"  Modes:         {', '.join(MODES)}")
print()
print(f"  {'Metric':<28} " + "  ".join(f"{m:>10}" for m in MODES))
print("  " + "-"*(28 + 14*len(MODES)))
all_metrics = (
    [("context_relevance", retrieval_dfs), ("evidence_recall", retrieval_dfs)]
    + [(m, generation_dfs) for m in ["rouge_l","answer_accuracy","faithfulness_paper","evidence_coverage"]]
)
for metric, dfs_dict in all_metrics:
    row = f"  {metric:<28} "
    for mode in MODES:
        row += f"  {dfs_dict[mode][metric].mean():>10.4f}"
    print(row)
print()
for col in ragas_num_cols:
    row = f"  ragas_{col:<22} "
    for mode in MODES:
        row += f"  {ragas_dfs[mode][col].mean():>10.4f}"
    print(row)